# Adaptive Quiz Generation: fine-tuning FLAN-T5-base for BTech MCQs

**Task:** given branch, subject, topic, difficulty and a context passage, generate one MCQ
(question, 4 options, answer letter, explanation).

**Data:** 11.9k cleaned MCQs (CSE / DSAI / ECE, 100 subjects, 2,000 topics), split **by topic**,
so test topics never appear in training.

**What this notebook does**
1. Loads the cleaned splits and runs EDA (token lengths).
2. Fixes the T5 tokenizer: adds the characters it can't encode (`{ } < ~ ^ \``), which our maths questions need.
3. Full fine-tuning of `google/flan-t5-base` with AdamW, cosine schedule plus warmup, label smoothing,
   gradient accumulation, early stopping and checkpoints every epoch (optionally pushed to the HF Hub, so a crash can't wipe the run).
4. Evaluates **baseline (zero-shot FLAN-T5) vs fine-tuned** on the test split:
   format validity, ROUGE-L, BLEU and difficulty control.
5. Saves the final model plus a zip for the app.

**Kaggle settings:** Accelerator = GPU T4 x2 (or P100), Internet = ON.
Add the dataset zip as input. Optionally add a Kaggle secret `HF_TOKEN` (write access) for Hub backups.
For a long run use **Save Version → Save & Run All (Commit)**, so it keeps running when you close the tab.

In [ ]:
!pip install -q -U "transformers>=4.46,<5" "datasets>=2.20" evaluate rouge_score sacrebleu peft accelerate

In [ ]:
import os, glob, json, math, random, re, time
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"   # less memory fragmentation
import numpy as np, pandas as pd, torch
from datasets import load_dataset
from transformers import (AutoTokenizer, AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq,
                          Seq2SeqTrainingArguments, Seq2SeqTrainer, EarlyStoppingCallback, set_seed)
print(torch.__version__, torch.cuda.is_available(), torch.cuda.device_count())

## 1. Config (hyperparameters)

In [ ]:
CFG = dict(
    model_name="google/flan-t5-base",
    out_dir="/kaggle/working/flan-t5-btech-mcq",   # the strategy name is appended below
    seed=42,
    # optimisation
    learning_rate=3e-4,          # standard for T5-base full fine-tuning with AdamW
    weight_decay=0.01,
    warmup_ratio=0.05,
    lr_scheduler="cosine",
    epochs=8,                    # upper bound, early stopping usually stops earlier
    per_device_batch=4,          # 8 ran out of memory on a 16 GB T4 (DataParallel gathers logits on GPU 0)
    effective_batch=32,          # per_device_batch * n_gpus * grad_accum
    label_smoothing=0.1,         # regularisation for a small synthetic dataset
    max_grad_norm=1.0,
    gradient_checkpointing=True, # recompute activations in backward: much less memory, ~25% slower
    early_stopping_patience=2,
    # lengths (re-computed from data below; these are caps)
    max_source_len=512,
    max_target_len=256,
    # fine-tuning strategy (run each once for the comparison table):
    #   "full"    : train all weights (main run)
    #   "freeze"  : freeze the lower `freeze_encoder_layers` encoder blocks, train the rest
    #   "lora"    : freeze everything, train small LoRA adapters (~2% params); use learning_rate=1e-3
    strategy="full",
    freeze_encoder_layers=6,     # flan-t5-base has 12 encoder blocks
    lora_r=16, lora_alpha=32, lora_dropout=0.05,
    # generation for evaluation
    num_beams=4,
    eval_samples=400,            # test examples for generation metrics (full test = slower)
    hub_model_id=None,           # e.g. "your-username/flan-t5-btech-mcq" to back up to the HF Hub
)
CFG["out_dir"] = f"{CFG['out_dir']}-{CFG['strategy']}"
n_gpu = max(1, torch.cuda.device_count())
CFG["grad_accum"] = max(1, CFG["effective_batch"] // (CFG["per_device_batch"] * n_gpu))
print(f"{n_gpu} GPU(s): batch {CFG['per_device_batch']} x {n_gpu} x accum {CFG['grad_accum']} = {CFG['per_device_batch']*n_gpu*CFG['grad_accum']}")
set_seed(CFG["seed"])

## 2. Load the cleaned dataset

In [ ]:
def find(name):
    hits = glob.glob(f"/kaggle/input/**/clean/{name}.jsonl", recursive=True) or \
           glob.glob(f"/kaggle/input/**/{name}.jsonl", recursive=True)
    assert hits, f"{name}.jsonl not found under /kaggle/input"
    return hits[0]

files = {s: find(s) for s in ["train", "val", "test"]}
print(files)
raw = load_dataset("json", data_files=files)
print(raw)
df = raw["train"].to_pandas()
print(df.groupby(["branch"]).size(), df.groupby("difficulty").size(), df.answer.value_counts(), sep="\n\n")

## 3. Tokenizer: add missing symbols and check token lengths

In [ ]:
tok = AutoTokenizer.from_pretrained(CFG["model_name"])
special = sorted({c for s in list(raw["train"]["input_text"]) + list(raw["train"]["target_text"]) for c in s
                  if not c.isalnum() and not c.isspace()})
missing = [c for c in special if tok.unk_token_id in tok(c, add_special_tokens=False).input_ids]
print("characters T5 cannot encode:", missing)
if missing:
    tok.add_tokens(missing)

src_len = [len(x) for x in tok(list(raw["train"]["input_text"])).input_ids]
tgt_len = [len(x) for x in tok(list(raw["train"]["target_text"])).input_ids]
for n, L in [("source", src_len), ("target", tgt_len)]:
    print(f"{n}: mean {np.mean(L):.0f}  p95 {np.percentile(L,95):.0f}  p99 {np.percentile(L,99):.0f}  max {max(L)}")
CFG["max_source_len"] = int(min(CFG["max_source_len"], math.ceil(max(src_len) / 8) * 8))
CFG["max_target_len"] = int(min(CFG["max_target_len"], math.ceil(max(tgt_len) / 8) * 8))
print("using", CFG["max_source_len"], CFG["max_target_len"])

## 4. Prompt format + parser (same code the app uses)

In [ ]:
import re

LETTERS = "ABCD"


def build_input(branch, subject, topic, difficulty, context):
    return (f"generate mcq | branch: {branch} | subject: {subject} | "
            f"topic: {topic} | difficulty: {difficulty} | context: {context}")


_PATTERN = re.compile(
    r"question:\s*(?P<q>.+?)\s*options:\s*"
    r"\(A\)\s*(?P<A>.+?)\s*\(B\)\s*(?P<B>.+?)\s*\(C\)\s*(?P<C>.+?)\s*\(D\)\s*(?P<D>.+?)\s*"
    r"answer:\s*(?P<ans>[ABCD])\b\s*(?:explanation:\s*(?P<exp>.*))?$",
    re.S)


def parse_mcq(text):
    """Parse model output into a dict, or return None if the format is broken."""
    m = _PATTERN.search(text.strip())
    if not m:
        return None
    options = [m.group(l).strip() for l in LETTERS]
    norm = [re.sub(r"\s+", " ", o.lower()) for o in options]
    if any(not o for o in options) or len(set(norm)) < 4:
        return None
    return {"question": m.group("q").strip(), "options": options,
            "answer": m.group("ans"), "explanation": (m.group("exp") or "").strip()}

## 5. Tokenise

In [ ]:
def preprocess(batch):
    enc = tok(batch["input_text"], max_length=CFG["max_source_len"], truncation=True)
    enc["labels"] = tok(text_target=batch["target_text"], max_length=CFG["max_target_len"], truncation=True)["input_ids"]
    return enc

keep = ["input_ids", "attention_mask", "labels"]
tokenized = raw.map(preprocess, batched=True, remove_columns=[c for c in raw["train"].column_names])
print(tokenized)

## 6. Model

In [ ]:
model = AutoModelForSeq2SeqLM.from_pretrained(CFG["model_name"])
if missing:
    model.resize_token_embeddings(len(tok))
if CFG["gradient_checkpointing"]:
    model.config.use_cache = False
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
CFG["use_lora"] = CFG["strategy"] == "lora"
if CFG["strategy"] == "freeze":
    for block in model.encoder.block[:CFG["freeze_encoder_layers"]]:
        for p in block.parameters():
            p.requires_grad = False
if CFG["use_lora"]:
    from peft import LoraConfig, get_peft_model, TaskType
    model = get_peft_model(model, LoraConfig(task_type=TaskType.SEQ_2_SEQ_LM, r=CFG["lora_r"],
                           lora_alpha=CFG["lora_alpha"], lora_dropout=CFG["lora_dropout"],
                           target_modules=["q", "k", "v", "o", "wi_0", "wi_1", "wo"],
                           modules_to_save=["shared", "lm_head"] if missing else None))
    model.print_trainable_parameters()
total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"strategy={CFG['strategy']}  params {total/1e6:.0f}M  trainable {trainable/1e6:.1f}M ({100*trainable/total:.1f}%)")

## 6b. Smoke test (catches errors before the long run)

In [ ]:
collator = DataCollatorForSeq2Seq(tok, model=model, label_pad_token_id=-100)
dev = "cuda" if torch.cuda.is_available() else "cpu"
model.to(dev).train()
# memory test: one forward + backward on the LONGEST examples at the real per-GPU batch size
lens = np.array([len(a) + len(b) for a, b in zip(tokenized["train"]["input_ids"], tokenized["train"]["labels"])])
longest = [int(i) for i in np.argsort(-lens)[:CFG["per_device_batch"]]]
batch = {k: v.to(dev) for k, v in collator([tokenized["train"][i] for i in longest]).items()}
if dev == "cuda":
    torch.cuda.reset_peak_memory_stats()
loss = model(**batch).loss
print("initial loss:", round(loss.item(), 3))
assert torch.isfinite(loss), "loss is NaN/inf, check precision settings"
loss.backward()
if dev == "cuda":
    peak = torch.cuda.max_memory_allocated() / 2**30
    total = torch.cuda.get_device_properties(0).total_memory / 2**30
    adam = 2 * sum(p.numel() for p in model.parameters() if p.requires_grad) * 4 / 2**30
    need = peak + adam + 1.5    # + optimizer states + DataParallel logits gathered on GPU 0
    print(f"GPU memory: peak {peak:.1f} GB + optimizer {adam:.1f} GB + margin 1.5 GB = {need:.1f} GB of {total:.1f} GB")
    assert need < 0.95 * total, "will run out of memory: lower CFG['per_device_batch'] to 2"
print("label check:", tok.decode([t for t in tokenized["train"][0]["labels"] if t >= 0], skip_special_tokens=True)[:200])
print("parser check:", parse_mcq(raw["train"][0]["target_text"]) is not None)
model.zero_grad(set_to_none=True)
del batch, loss
if dev == "cuda":
    torch.cuda.empty_cache()

## 7. Train

- T5 is unstable in fp16 (activations overflow to NaN), so we train in **fp32**. bf16 is used automatically on GPUs that support it (A100/L4).
- A checkpoint is saved every epoch (best + latest kept). If `HF_TOKEN` and `hub_model_id` are set, checkpoints are also pushed to the Hub.
- If the session restarts, re-run: training **resumes from the last checkpoint**.

In [ ]:
hub_token = None
try:
    from kaggle_secrets import UserSecretsClient
    hub_token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass
push = bool(hub_token and CFG["hub_model_id"])
# real bf16 needs compute capability >= 8 (A100/L4); T4 and P100 must use fp32
bf16 = torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8

args = Seq2SeqTrainingArguments(
    output_dir=CFG["out_dir"],
    num_train_epochs=CFG["epochs"],
    learning_rate=CFG["learning_rate"],
    weight_decay=CFG["weight_decay"],
    warmup_ratio=CFG["warmup_ratio"],
    lr_scheduler_type=CFG["lr_scheduler"],
    per_device_train_batch_size=CFG["per_device_batch"],
    per_device_eval_batch_size=CFG["per_device_batch"],
    gradient_accumulation_steps=CFG["grad_accum"],
    label_smoothing_factor=CFG["label_smoothing"],
    max_grad_norm=CFG["max_grad_norm"],
    gradient_checkpointing=CFG["gradient_checkpointing"],
    gradient_checkpointing_kwargs={"use_reentrant": False},
    optim="adamw_torch",
    bf16=bf16, fp16=False,
    eval_strategy="epoch", save_strategy="epoch", logging_steps=25,
    save_total_limit=1, load_best_model_at_end=True,   # keeps best + latest (~3 GB each)
    metric_for_best_model="eval_loss", greater_is_better=False,
    group_by_length=True,            # batches of similar length = less padding = faster
    predict_with_generate=False,     # generation metrics are computed after training
    dataloader_num_workers=2,
    report_to="none", seed=CFG["seed"],
    push_to_hub=push, hub_model_id=CFG["hub_model_id"], hub_token=hub_token,
    hub_strategy="every_save", hub_private_repo=True,
)
trainer = Seq2SeqTrainer(
    model=model, args=args,
    train_dataset=tokenized["train"], eval_dataset=tokenized["val"],
    data_collator=DataCollatorForSeq2Seq(tok, model=model, label_pad_token_id=-100),
    processing_class=tok,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=CFG["early_stopping_patience"])],
)
last = None
if os.path.isdir(CFG["out_dir"]):
    ck = sorted(glob.glob(f"{CFG['out_dir']}/checkpoint-*"), key=lambda p: int(p.split("-")[-1]))
    last = ck[-1] if ck else None
print("resuming from", last)
t0 = time.time()
train_result = trainer.train(resume_from_checkpoint=last)
print(f"training took {(time.time()-t0)/60:.1f} min")

In [ ]:
print(trainer.evaluate())
final_dir = f"{CFG['out_dir']}/final"
model_to_save = trainer.model.merge_and_unload() if CFG["use_lora"] else trainer.model
model_to_save.config.use_cache = True   # was turned off for gradient checkpointing; needed for fast generation
model_to_save.save_pretrained(final_dir, safe_serialization=True)
tok.save_pretrained(final_dir)
json.dump(CFG, open(f"{final_dir}/train_config.json", "w"), indent=2)
if push:
    trainer.push_to_hub(commit_message="final model")

## 8. Training curves

In [ ]:
import matplotlib.pyplot as plt
h = pd.DataFrame(trainer.state.log_history)
tr = h.dropna(subset=["loss"]); ev = h.dropna(subset=["eval_loss"])
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(tr["step"], tr["loss"], label="train"); ax[0].plot(ev["step"], ev["eval_loss"], "o-", label="val")
ax[0].set_title("Loss"); ax[0].set_xlabel("step"); ax[0].legend()
ax[1].plot(tr["step"], tr["learning_rate"]); ax[1].set_title("Learning rate (warmup + cosine)"); ax[1].set_xlabel("step")
plt.tight_layout(); plt.savefig(f"{CFG['out_dir']}/training_curves.png", dpi=150); plt.show()

## 9. Evaluation: baseline vs fine-tuned (test split, unseen topics)

| Metric | What it measures |
|---|---|
| format_valid | output parses into question, 4 distinct options, an answer letter and an explanation |
| rougeL_question | overlap of the generated question with the reference question |
| bleu_full | BLEU of the whole generated MCQ vs the reference |
| avg_q_words by difficulty | **difficulty control**: hard questions should come out longer and more complex than easy ones |

The baseline is the original `flan-t5-base`, given an instruction that spells out the output format (so the comparison is fair).

In [ ]:
import evaluate
rouge, bleu = evaluate.load("rouge"), evaluate.load("sacrebleu")
test = raw["test"].shuffle(seed=0).select(range(min(CFG["eval_samples"], len(raw["test"]))))
dev = "cuda" if torch.cuda.is_available() else "cpu"

BASELINE_INSTR = ("Write one multiple-choice question from the context. Use exactly this format: "
                  "question: <question> options: (A) <option> (B) <option> (C) <option> (D) <option> "
                  "answer: <letter> explanation: <why>. ")

@torch.no_grad()
def generate(m, t, prompts, bs=16, **gen):
    m.eval().to(dev); outs = []
    for i in range(0, len(prompts), bs):
        enc = t(prompts[i:i+bs], return_tensors="pt", padding=True, truncation=True,
                max_length=CFG["max_source_len"]).to(dev)
        ids = m.generate(**enc, max_new_tokens=CFG["max_target_len"], **gen)
        outs += t.batch_decode(ids, skip_special_tokens=True)
    return outs

def score(preds, ds, name):
    parsed = [parse_mcq(p) for p in preds]
    ok = [p for p in parsed if p]
    refs_q = [r for p, r in zip(parsed, list(ds["target_text"])) if p]
    res = {"model": name, "format_valid_%": round(100 * len(ok) / len(preds), 1)}
    gen_q = [p["question"] for p in ok]
    ref_q = [parse_mcq(r)["question"] for r in refs_q]
    res["rougeL_question"] = round(rouge.compute(predictions=gen_q, references=ref_q)["rougeL"], 4) if ok else 0
    res["bleu_full"] = round(bleu.compute(predictions=preds, references=[[r] for r in list(ds["target_text"])])["score"], 2)
    for d in ["easy", "medium", "hard"]:
        q = [p["question"] for p, dd in zip(parsed, list(ds["difficulty"])) if p and dd == d]
        res[f"avg_q_words_{d}"] = round(np.mean([len(x.split()) for x in q]), 1) if q else None
    return res

results = []
base_tok = AutoTokenizer.from_pretrained(CFG["model_name"])
base = AutoModelForSeq2SeqLM.from_pretrained(CFG["model_name"])
base_preds = generate(base, base_tok, [BASELINE_INSTR + x for x in list(test["input_text"])], num_beams=CFG["num_beams"])
results.append(score(base_preds, test, "flan-t5-base (zero-shot)"))
del base; torch.cuda.empty_cache()

ft = AutoModelForSeq2SeqLM.from_pretrained(final_dir)
ft_tok = AutoTokenizer.from_pretrained(final_dir)
ft_preds = generate(ft, ft_tok, list(test["input_text"]), num_beams=CFG["num_beams"])
results.append(score(ft_preds, test, "flan-t5-base (fine-tuned)"))

ref = {"model": "reference (dataset)"}
for d in ["easy", "medium", "hard"]:
    ref[f"avg_q_words_{d}"] = round(np.mean([len(parse_mcq(t)["question"].split())
                                             for t, dd in zip(list(test["target_text"]), list(test["difficulty"])) if dd == d]), 1)
results.append(ref)
res_df = pd.DataFrame(results); display(res_df)
res_df.to_csv(f"{CFG['out_dir']}/results.csv", index=False)
pd.DataFrame({"input": list(test["input_text"]), "reference": list(test["target_text"]), "baseline": base_preds,
              "fine_tuned": ft_preds}).to_csv(f"{CFG['out_dir']}/test_predictions.csv", index=False)

## 9b. Difficulty classifier + grounding check (ideas carried over from last year's report)

Last year's report planned to *classify generated question difficulty* and *filter by similarity to the source text*.
Both are used here as **extra evaluation metrics**:

- **difficulty_match_%**: a TF-IDF + Logistic Regression classifier is trained on the train-split MCQs to predict easy/medium/hard.
  It is then run on each model's outputs. A higher match with the requested difficulty means better difficulty control.
  The classifier's accuracy on the reference test MCQs is the ceiling; labels are noisy, so it won't be near 100%.
- **grounding**: the share of content words in the question + correct option that also appear in the context passage.
  Low values mean the model is making things up instead of using the passage.

The app can reuse both for filtering: the classifier is saved as `difficulty_clf.joblib`.

In [ ]:
import re, joblib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, classification_report

def mcq_text(p):
    return p["question"] + " || " + " | ".join(p["options"])

tr_parsed = [parse_mcq(t) for t in list(raw["train"]["target_text"])]
X_tr = [mcq_text(p) for p in tr_parsed if p]
y_tr = [d for p, d in zip(tr_parsed, list(raw["train"]["difficulty"])) if p]
diff_clf = make_pipeline(TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True),
                         LogisticRegression(max_iter=2000, C=2.0))
diff_clf.fit(X_tr, y_tr)
ref_parsed = [parse_mcq(t) for t in list(test["target_text"])]
print("classifier accuracy on reference test MCQs (ceiling):",
      round(accuracy_score(list(test["difficulty"]), diff_clf.predict([mcq_text(p) for p in ref_parsed])), 3))
joblib.dump(diff_clf, f"{CFG['out_dir']}/difficulty_clf.joblib")

STOP = set("the a an of to in is are and or for on with by as at be this that which from it its was were what how why when".split())
def content(s): return {w for w in re.findall(r"[a-z0-9]+", s.lower()) if w not in STOP and len(w) > 2}

def extra_scores(preds, ds):
    parsed = [parse_mcq(p) for p in preds]
    pairs = [(p, d, x) for p, d, x in zip(parsed, list(ds["difficulty"]), list(ds["input_text"])) if p]
    if not pairs:
        return {"difficulty_match_%": 0, "grounding": 0}
    pred_d = diff_clf.predict([mcq_text(p) for p, _, _ in pairs])
    match = np.mean([a == b for a, (_, b, _) in zip(pred_d, pairs)])
    g = []
    for p, _, x in pairs:
        ctx = content(x.split("context:", 1)[-1])
        ans = p["options"]["ABCD".index(p["answer"])]
        w = content(p["question"] + " " + ans)
        g.append(len(w & ctx) / max(1, len(w)))
    return {"difficulty_match_%": round(100 * match, 1), "grounding": round(float(np.mean(g)), 3)}

for row, preds in [(results[0], base_preds), (results[1], ft_preds), (results[2], list(test["target_text"]))]:
    row.update(extra_scores(preds, test))
res_df = pd.DataFrame(results); display(res_df)
res_df.to_csv(f"{CFG['out_dir']}/results.csv", index=False)

## 10. Try it: generate fresh questions (sampling for variety)

In [ ]:
def generate_mcq(branch, subject, topic, difficulty, context, n=3):
    prompt = build_input(branch, subject, topic, difficulty, context)
    enc = ft_tok([prompt], return_tensors="pt", truncation=True, max_length=CFG["max_source_len"]).to(dev)
    ids = ft.generate(**enc, do_sample=True, top_p=0.92, temperature=0.8, num_return_sequences=n * 2,
                      max_new_tokens=CFG["max_target_len"], no_repeat_ngram_size=4)
    out, seen = [], set()
    for t in ft_tok.batch_decode(ids, skip_special_tokens=True):
        p = parse_mcq(t)
        if p and p["question"].lower() not in seen:
            seen.add(p["question"].lower()); out.append(p)
    return out[:n]

r = raw["test"][0]
for d in ["easy", "medium", "hard"]:
    print(f"--- {d} ---")
    for q in generate_mcq(r["branch"], r["subject"], r["topic"], d, r["context"], n=2):
        print(q["question"]); [print(f"  ({l}) {o}") for l, o in zip("ABCD", q["options"])]
        print("  answer:", q["answer"], "|", q["explanation"])

## 11. Package the model for the app (download from the Output tab)

In [ ]:
import shutil
shutil.make_archive("/kaggle/working/flan_t5_btech_mcq_model", "zip", final_dir)
print(os.path.getsize("/kaggle/working/flan_t5_btech_mcq_model.zip") / 1e6, "MB")